# Group 13 ANN Experiments — Phase C verification

This notebook verifies an **uncommitted Windows working-tree snapshot** of Phase C. It does not clone the older GitHub feature branch for source code. Upload the `phase-c-working-tree.zip` artifact supplied for this run. The manifest identifies the base Git commit, exported file hashes, and a content-derived snapshot ID.

The notebook separately checks out the approved SRSD-Feynman Medium dataset at its pinned Hugging Face revision. It validates the dataset before tests, runs the symbolic tests, data tests, production smoke test, and full suite in order, and stops at the first failed, skipped, or blocked stage. Results and the snapshot identity are saved to Drive. No Phase D experiments are run.


## 1. Environment setup and hardware check

In [ ]:
import os, platform, random, subprocess, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
except ImportError as exc:
    raise RuntimeError('Run this notebook in Google Colab.') from exc

print('Python:', platform.python_version())
try:
    import torch
except ImportError as exc:
    raise RuntimeError('PyTorch is missing. Select a standard Colab runtime; this notebook avoids reinstalling the large Colab PyTorch build.') from exc

print('PyTorch:', torch.__version__)
print('CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    subprocess.run(['nvidia-smi'], check=False)
else:
    print('GPU: none (CPU execution is available, but training will be slower)')

SEED = 13
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device selected:', DEVICE, '| Seed:', SEED)

## 2. Google Drive and persistent project folders

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/ann-group13')
CANONICAL_DATA_ROOT = Path('/content/drive/MyDrive/ann-group13/data/srsd-feynman_medium')
DATA_ROOT = CANONICAL_DATA_ROOT
RESULTS_DIR = DRIVE_ROOT / 'results'
CHECKPOINTS_DIR = DRIVE_ROOT / 'checkpoints'
FIGURES_DIR = DRIVE_ROOT / 'figures'
for folder in (DATA_ROOT.parent, RESULTS_DIR, CHECKPOINTS_DIR, FIGURES_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print('Drive project directory:', DRIVE_ROOT)
print('Canonical dataset directory:', DATA_ROOT)
print('Persistent outputs:', RESULTS_DIR, CHECKPOINTS_DIR, FIGURES_DIR)

## 3. Transfer the exact uncommitted working tree

In Google Colab, run this notebook from a fresh runtime and upload the supplied `phase-c-working-tree.zip` in the next cell. That ZIP was exported from the current `phase-c-verification` checkout, includes tracked working-tree contents and untracked project files (including `tests/test_data.py`), excludes `.git` and the dataset payload, and carries a SHA-256 manifest. The notebook verifies every file hash and prints `UNCOMMITTED_WORKTREE_SNAPSHOT` plus its snapshot ID. Upload that exact file; do not substitute a GitHub branch checkout.

In [ ]:
import hashlib, importlib.util, importlib.metadata, json, shutil, zipfile
from pathlib import PurePosixPath
from google.colab import files

uploads = files.upload()
zip_names = [name for name in uploads if name.lower().endswith('.zip')]
assert len(zip_names) == 1, f'Upload exactly one phase-c-working-tree ZIP; received {list(uploads)}'
SNAPSHOT_ZIP = Path(zip_names[0])
REPO = Path('/content/ann-group13-phase-c-snapshot')
assert not REPO.exists(), f'{REPO} already exists; use a fresh Colab runtime to avoid mixing snapshots.'
REPO.mkdir(parents=True)
with zipfile.ZipFile(SNAPSHOT_ZIP) as zf:
    members = zf.namelist()
    assert len(members) == len(set(members)), 'ZIP contains duplicate paths'
    for member in members:
        rel = PurePosixPath(member)
        assert '\\' not in member and not rel.is_absolute() and '..' not in rel.parts, f'Unsafe ZIP member: {member}'
        info = zf.getinfo(member)
        mode = (info.external_attr >> 16) & 0o170000
        assert mode != 0o120000, f'Symlink ZIP member is not allowed: {member}'
    zf.extractall(REPO)

manifest_path = REPO / 'COLAB_SOURCE_MANIFEST.json'
assert manifest_path.is_file(), 'Snapshot ZIP is missing COLAB_SOURCE_MANIFEST.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest.get('revision_kind') == 'uncommitted_working_tree_snapshot'
assert manifest.get('branch') == 'phase-c-verification'
manifest_paths = [entry['path'] for entry in manifest['files']]
assert manifest_paths == sorted(manifest_paths) and len(manifest_paths) == len(set(manifest_paths)), 'Manifest paths must be unique and sorted'
for rel_path in manifest_paths:
    rel = PurePosixPath(rel_path)
    assert '\\' not in rel_path and not rel.is_absolute() and '..' not in rel.parts, f'Unsafe manifest path: {rel_path}'
expected_members = set(manifest_paths) | {'COLAB_SOURCE_MANIFEST.json'}
assert set(members) == expected_members, 'ZIP members do not exactly match the manifest'
for entry in manifest['files']:
    rel = PurePosixPath(entry['path'])
    source_path = REPO.joinpath(*rel.parts)
    assert source_path.is_file(), f"Snapshot file missing: {entry['path']}"
    actual = hashlib.sha256(source_path.read_bytes()).hexdigest()
    assert actual == entry['sha256'], f"Snapshot hash mismatch: {entry['path']}"
hash_input = ''.join(f"{f['path']}\0{f['sha256']}\n" for f in manifest['files']).encode('utf-8')
actual_snapshot_id = hashlib.sha256(hash_input).hexdigest()
assert actual_snapshot_id == manifest['snapshot_id'], 'Snapshot content ID mismatch'

BASE_COMMIT = manifest['base_commit']
SNAPSHOT_ID = manifest['snapshot_id']
CODE_REVISION_KIND = 'UNCOMMITTED_WORKTREE_SNAPSHOT'
assert (REPO / 'tests' / 'test_data.py').is_file(), 'Transferred snapshot lacks tests/test_data.py'
assert 'def load_problem(' in (REPO / 'src' / 'data.py').read_text(encoding='utf-8')
assert 'def test_to_original_float_forward_regressions_against_ground_truth' in (REPO / 'tests' / 'test_symbolic.py').read_text(encoding='utf-8')
os.chdir(REPO)
print('Source revision kind:', CODE_REVISION_KIND)
print('Base Git commit (provenance only):', BASE_COMMIT)
print('Working-tree snapshot ID:', SNAPSHOT_ID)
print('Exported files:', len(manifest['files']))
print('Export status at transfer:', manifest['git_status'])

# Install the project requirements and pytest; leave the Colab PyTorch/NumPy stack intact.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements.txt')], check=True)
if importlib.util.find_spec('pytest') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest'], check=True)
import numpy, sympy, pandas, matplotlib, pytest
import zss, kan, torch
print('Python:', platform.python_version())
print('NumPy:', numpy.__version__)
print('SymPy:', sympy.__version__)
print('PyTorch:', torch.__version__)
print('pytest:', pytest.__version__)
print('zss:', importlib.metadata.version('zss'))
print('pykan:', importlib.metadata.version('pykan'))


## 4. Download and validate the pinned SRSD-Feynman Medium dataset

The approved dataset is checked out at Hugging Face revision `de7af9e2638494678ebe18d1174bbc6a02037eec`. The validation cell checks the exact checkout hash, all 40 ground-truth equation IDs, matching train/val/test file stems, split row counts, numeric finiteness, and consistent 1–6 feature dimensions. Any discrepancy stops execution before tests.


In [ ]:
import pickle
import numpy as np

DATASET_URL = 'https://huggingface.co/datasets/yoshitomo-matsubara/srsd-feynman_medium'
DATASET_REVISION = 'de7af9e2638494678ebe18d1174bbc6a02037eec'
if not shutil.which('git-lfs'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'git-lfs'], check=True)
subprocess.run(['git', 'lfs', 'install'], check=True)
if not DATA_ROOT.exists():
    subprocess.run(['git', 'clone', '--no-checkout', DATASET_URL, str(DATA_ROOT)], check=True)
elif not (DATA_ROOT / '.git').exists():
    raise RuntimeError(f'{DATA_ROOT} exists but is not the expected dataset checkout. Inspect it before replacing it.')

remote_url = subprocess.check_output(
    ['git', '-C', str(DATA_ROOT), 'remote', 'get-url', 'origin'], text=True
).strip().rstrip('/')
assert remote_url in {DATASET_URL, DATASET_URL + '.git'}, f'Unexpected dataset remote: {remote_url}'
subprocess.run(['git', '-C', str(DATA_ROOT), 'fetch', 'origin', DATASET_REVISION], check=True)
subprocess.run(['git', '-C', str(DATA_ROOT), 'checkout', '--detach', DATASET_REVISION], check=True)
subprocess.run(['git', '-C', str(DATA_ROOT), 'lfs', 'pull', 'origin'], check=True)
DATASET_CHECKOUT = subprocess.check_output(
    ['git', '-C', str(DATA_ROOT), 'rev-parse', 'HEAD'], text=True
).strip()
assert DATASET_CHECKOUT == DATASET_REVISION, (
    f'Expected dataset revision {DATASET_REVISION}, found {DATASET_CHECKOUT}'
)
print('Pinned Medium dataset revision:', DATASET_CHECKOUT)

split_dirs = {split: DATA_ROOT / split for split in ('train', 'val', 'test')}
eq_dir = DATA_ROOT / 'true_eq'
assert all(path.is_dir() for path in split_dirs.values()), 'Missing train/val/test directory'
assert eq_dir.is_dir(), 'Missing true_eq directory'
equations = {p.stem for p in eq_dir.glob('*.pkl')}
assert len(equations) == 40, f'Expected 40 Medium equations; found {len(equations)}'
feature_dims = {split: {} for split in ('train', 'val', 'test')}
for split, expected_rows in (('train', 8000), ('val', 1000), ('test', 1000)):
    files = {p.stem: p for p in split_dirs[split].glob('*.txt')}
    assert set(files) == equations, f'{split} equation IDs differ from true_eq'
    for name, path in files.items():
        arr = np.loadtxt(path, dtype=np.float64)
        assert arr.ndim == 2 and arr.shape[0] == expected_rows, f'{path.name}: unexpected shape {arr.shape}'
        d = arr.shape[1] - 1
        assert 1 <= d <= 6, f'{path.name}: expected 1–6 input columns, got {d}'
        feature_dims[split][path.stem] = d
        assert np.isfinite(arr).all(), f'{path.name}: non-finite values found'
    print(f'{split}: {len(files)} files, {expected_rows} rows per file')

for name in equations:
    dims = {split: feature_dims[split][name] for split in ('train', 'val', 'test')}
    assert len(set(dims.values())) == 1, f'{name}: input dimensions differ across splits: {dims}'

# Check all ground-truth pickles are readable and target standardization remains finite.
for name in sorted(equations):
    with (eq_dir / f'{name}.pkl').open('rb') as f:
        _ = pickle.load(f)
    y_train = np.loadtxt(split_dirs['train'] / f'{name}.txt', dtype=np.float64)[:, -1]
    mean, std = y_train.mean(dtype=np.float64), y_train.std(dtype=np.float64)
    assert np.isfinite(mean) and np.isfinite(std) and std > 0, f'{name}: invalid train target statistics'
    for split in ('train', 'val', 'test'):
        y = np.loadtxt(split_dirs[split] / f'{name}.txt', dtype=np.float64)[:, -1]
        y_scaled = ((y - mean) / std).astype(np.float32)
        assert np.isfinite(y_scaled).all(), f'{name}/{split}: target scaling overflowed float32'

RESOLVED_DATA_ROOT = DATA_ROOT.resolve(strict=True)
assert RESOLVED_DATA_ROOT == CANONICAL_DATA_ROOT.resolve(strict=True), (
    f'Expected canonical Drive dataset at {CANONICAL_DATA_ROOT}, found {RESOLVED_DATA_ROOT}'
)
os.environ['SRSD_ROOT'] = str(RESOLVED_DATA_ROOT)
assert Path(os.environ['SRSD_ROOT']).resolve(strict=True) == RESOLVED_DATA_ROOT
print('Validated Medium dataset:', len(equations), 'equations at', RESOLVED_DATA_ROOT)
print('SRSD_ROOT:', os.environ['SRSD_ROOT'])

## 5. Phase C verification sequence

The next cell runs exactly these stages, in order: `pytest tests/test_symbolic.py`, `pytest tests/test_data.py`, `python scripts/phase_c_data_smoke.py`, then `pytest` for the full suite. It records each stage's actual command, output, return code, and pytest counts. It stops immediately on a failing command, a missing test summary, any failure/error, or any skipped/xfailed test; none are labeled passed. The partial record is written to Drive before stopping.


In [ ]:
import datetime
import json
import re

os.chdir(REPO)
EXPECTED_DATA_ROOT = CANONICAL_DATA_ROOT.resolve(strict=True)
assert DATA_ROOT.resolve(strict=True) == EXPECTED_DATA_ROOT
configured_srsd_root = os.environ.get('SRSD_ROOT')
assert configured_srsd_root is not None and Path(configured_srsd_root).resolve(strict=True) == EXPECTED_DATA_ROOT, (
    f'SRSD_ROOT must resolve to the canonical Drive dataset {EXPECTED_DATA_ROOT}; '
    f'found {configured_srsd_root!r}'
)
verification_env = os.environ.copy()
verification_env['SRSD_ROOT'] = str(EXPECTED_DATA_ROOT)
verification_commands = [
    ('symbolic_tests', [sys.executable, '-m', 'pytest', '-q', '-rs', 'tests/test_symbolic.py']),
    ('data_tests', [sys.executable, '-m', 'pytest', '-q', '-rs', 'tests/test_data.py']),
    ('production_smoke', [sys.executable, 'scripts/phase_c_data_smoke.py']),
    ('full_suite', [sys.executable, '-m', 'pytest', '-q', '-rs']),
]
record = {
    'timestamp_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'source_revision_kind': CODE_REVISION_KIND,
    'source_snapshot_id': SNAPSHOT_ID,
    'base_git_commit': BASE_COMMIT,
    'dataset_revision': DATASET_REVISION,
    'dataset_checkout': DATASET_CHECKOUT,
    'dataset_root': str(EXPECTED_DATA_ROOT),
    'srsd_root_env': verification_env['SRSD_ROOT'],
    'device': DEVICE,
    'steps': [],
}
record_path = RESULTS_DIR / f'phase_c_{SNAPSHOT_ID[:12]}_verification.json'

def save_record():
    record_path.write_text(json.dumps(record, indent=2), encoding='utf-8')

def pytest_counts(text):
    counts = {key: 0 for key in ('passed', 'failed', 'errors', 'skipped', 'xfailed', 'xpassed')}
    for key in counts:
        matches = re.findall(rf'(?m)(\d+) {key}\b', text)
        if matches:
            counts[key] = int(matches[-1])
    summary = [line.strip() for line in text.splitlines()
               if re.search(r'\b\d+ (?:passed|failed|skipped|error|errors|xfailed|xpassed)\b', line)]
    return counts, summary

for name, command in verification_commands:
    print(f'\n===== {name}: {" ".join(command)} =====', flush=True)
    run = subprocess.run(command, text=True, capture_output=True, env=verification_env)
    combined = run.stdout + ('\n' + run.stderr if run.stderr else '')
    print(combined, end='' if combined.endswith('\n') else '\n')
    counts, summary = pytest_counts(combined) if 'pytest' in command else ({}, [])
    if run.returncode != 0:
        status = 'blocked' if 'No module named pytest' in combined or 'No module named' in combined else 'failed'
    elif 'pytest' in command and (not summary or any(counts[k] for k in ('failed', 'errors', 'skipped', 'xfailed', 'xpassed'))):
        status = 'blocked' if not summary or counts['skipped'] or counts['xfailed'] else 'failed'
    else:
        status = 'passed'
    step = {
        'name': name,
        'command': command,
        'status': status,
        'returncode': run.returncode,
        'pytest_counts': counts,
        'pytest_summary_lines': summary,
        'stdout': run.stdout,
        'stderr': run.stderr,
    }
    record['steps'].append(step)
    save_record()
    print('Stage status:', status)
    if status != 'passed':
        raise RuntimeError(f'{name} {status}; stopped here. See captured output and {record_path}.')

print('All Phase C verification stages passed with no skips.')
print('Source snapshot:', SNAPSHOT_ID)
print('Verification record:', record_path)


## 6. Pilot experiments — deferred

Do not launch a pilot yet. The production SRSD data loader (`src/data.py:load_problem`) is now implemented, and its Phase C data and training smoke test passed in Colab with the pinned Medium dataset. Pilot experiments remain deferred until the remaining model and experiment components and final specifications are completed and reviewed. Once those are ready, add a small pilot here for CK-2 and CK-3, multiple seeds, CSV persistence, and checkpoint recovery.

## 7. Full experiments — deferred

Run the full Medium benchmark only after the pilot is reviewed and the official Yousef Plan confirms the final specifications. Keep all checkpoints and results under `DRIVE_ROOT`.

## 8. Results and figures — deferred

Create summaries and figures only from completed experiment outputs. No results or figures are generated by this setup notebook.